# 3D result Export

## Plot Field

In [1]:
import ansys.aedt.core
import os

import tempfile
import time
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.
from ansys.aedt.core import Desktop

import ansys.aedt.core
from ansys.aedt.core import Desktop
import subprocess
import psutil
import time
import os
import re

def get_aedt_processes_detailed():
    """
    실행 중인 AEDT 프로세스를 상세히 확인합니다.
    
    Returns:
    --------
    list : AEDT 프로세스 정보 리스트
    """
    print("🔍 실행 중인 AEDT 프로세스 검색...")
    
    aedt_processes = []
    try:
        for proc in psutil.process_iter(['pid', 'name', 'cmdline', 'create_time']):
            try:
                pinfo = proc.info
                process_name = pinfo['name'] if pinfo['name'] else ""
                
                # AEDT 관련 프로세스 필터링
                if any(keyword in process_name.lower() for keyword in ['ansysedt', 'aedt']):
                    # 포트 정보 추출 시도
                    ports = []
                    try:
                        connections = proc.connections()
                        for conn in connections:
                            if conn.status == 'LISTEN':
                                ports.append(conn.laddr.port)
                    except (psutil.AccessDenied, psutil.NoSuchProcess):
                        pass
                    
                    aedt_processes.append({
                        'pid': pinfo['pid'],
                        'name': process_name,
                        'cmdline': pinfo['cmdline'] if pinfo['cmdline'] else [],
                        'create_time': time.ctime(pinfo['create_time']),
                        'ports': ports
                    })
                    
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                continue
    
        if aedt_processes:
            print(f"✅ {len(aedt_processes)}개의 AEDT 프로세스 발견:")
            for i, proc in enumerate(aedt_processes):
                print(f"\n📋 프로세스 {i+1}:")
                print(f"   PID: {proc['pid']}")
                print(f"   이름: {proc['name']}")
                print(f"   생성시간: {proc['create_time']}")
                if proc['ports']:
                    print(f"   열린 포트: {proc['ports']}")
                else:
                    print(f"   열린 포트: 없음")
        else:
            print("❌ AEDT 프로세스가 없습니다.")
            
        return aedt_processes
        
    except Exception as e:
        print(f"❌ 프로세스 검색 중 오류: {e}")
        return []

def try_connect_to_existing_desktop():
    """
    기존 AEDT Desktop에 연결을 시도합니다.
    
    Returns:
    --------
    Desktop or None : 연결된 Desktop 객체 또는 None
    """
    print("🔗 기존 AEDT Desktop 연결 시도...")
    
    try:
        # 방법 1: new_desktop_session=False로 기존 세션에 연결
        desktop = Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=False,
            non_graphical=NG_MODE
        )
        print("✅ 기존 AEDT Desktop에 성공적으로 연결되었습니다!")
        return desktop
        
    except Exception as e:
        print(f"❌ 기존 Desktop 연결 실패: {e}")
        return None

def try_connect_with_ports(port_list):
    """
    특정 포트들을 시도해서 AEDT에 연결합니다.
    
    Parameters:
    -----------
    port_list : list
        시도할 포트 번호 리스트
        
    Returns:
    --------
    Desktop or None : 연결된 Desktop 객체 또는 None
    """
    AEDT_VERSION='251'
    NG_MODE=False
    for port in port_list:
        try:
            print(f"🔗 포트 {port}로 연결 시도...")
            desktop = Desktop(
                specified_version=AEDT_VERSION,
                new_desktop_session=False,
                port=port,
                non_graphical=NG_MODE
            )
            print(f"✅ 포트 {port}로 성공적으로 연결되었습니다!")
            return desktop
        except Exception as e:
            print(f"❌ 포트 {port} 연결 실패: {e}")
            continue
    
    return None

def get_desktop_connection():
    """
    다양한 방법으로 AEDT Desktop 연결을 시도합니다.
    
    Returns:
    --------
    Desktop : 연결된 Desktop 객체
    """
    print("=" * 60)
    print("🎯 AEDT Desktop 연결 시도")
    print("=" * 60)
    
    # 1. 기존 Desktop 연결 시도
    desktop = try_connect_to_existing_desktop()
    if desktop:
        return desktop
    
    # 2. 프로세스에서 포트 찾아서 연결 시도
    processes = get_aedt_processes_detailed()
    all_ports = []
    
    for proc in processes:
        all_ports.extend(proc['ports'])
    
    if all_ports:
        desktop = try_connect_with_ports(all_ports)
        if desktop:
            return desktop
    
    # 3. 일반적인 AEDT 포트들 시도
    common_ports = [56800, 56801, 56802, 56803, 56804, 56805]
    print("\n🔍 일반적인 AEDT 포트들 시도...")
    desktop = try_connect_with_ports(common_ports)
    if desktop:
        return desktop
    
    # 4. 새로운 Desktop 세션 생성
    print("\n🆕 새로운 AEDT Desktop 세션을 생성합니다...")
    try:
        desktop = Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=True,
            non_graphical=NG_MODE
        )
        print("✅ 새로운 AEDT Desktop이 생성되었습니다!")
        return desktop
    except Exception as e:
        print(f"❌ 새 Desktop 생성 실패: {e}")
        return None

def check_current_desktop_status(desktop):
    """
    현재 Desktop의 상태를 확인합니다.
    
    Parameters:
    -----------
    desktop : Desktop
        확인할 Desktop 객체
    """
    if not desktop:
        print("❌ Desktop 객체가 없습니다.")
        return
    
    try:
        print("\n" + "=" * 40)
        print("📊 현재 Desktop 상태:")
        print("=" * 40)
        
        # 기본 정보
        print(f"AEDT 버전: {desktop.aedt_version_id}")
        print(f"프로세스 ID: {desktop.aedt_process_id}")
        
        # 프로젝트 정보
        try:
            projects = desktop.project_list()
            print(f"\n📁 열린 프로젝트 ({len(projects)}개):")
            for i, proj_name in enumerate(projects, 1):
                print(f"  {i}. {proj_name}")
            
            # 활성 프로젝트
            active_proj = desktop.active_project()
            if active_proj:
                proj_name = active_proj.GetName()
                print(f"\n🎯 활성 프로젝트: {proj_name}")
                
                # 디자인 목록
                try:
                    design_list = active_proj.GetTopDesignList()
                    print(f"📐 디자인 ({len(design_list)}개):")
                    for i, design in enumerate(design_list, 1):
                        print(f"  {i}. {design}")
                        
                    # 활성 디자인
                    active_design = desktop.active_design()
                    if active_design:
                        print(f"🎯 활성 디자인: {active_design.GetName()}")
                        print(f"   디자인 타입: {active_design.GetDesignType()}")
                except:
                    print("디자인 정보 가져오기 실패")
            else:
                print("🎯 활성 프로젝트: 없음")
                
        except Exception as e:
            print(f"프로젝트 정보 가져오기 실패: {e}")
            
    except Exception as e:
        print(f"❌ Desktop 상태 확인 중 오류: {e}")

def smart_aedt_connector():
    """
    스마트 AEDT 연결 함수 - 사용자 친화적 인터페이스
    
    Returns:
    --------
    Desktop : 연결된 Desktop 객체
    """
    print("🚀 스마트 AEDT 연결기를 시작합니다...")
    
    # Desktop 연결 시도
    desktop = get_desktop_connection()
    
    if desktop:
        # 연결 성공 시 상태 확인
        check_current_desktop_status(desktop)
        
        print("\n" + "=" * 60)
        print("🎉 AEDT Desktop 연결이 완료되었습니다!")
        print("💡 다음과 같이 사용할 수 있습니다:")
        print("=" * 60)
        print("# 프로젝트 열기:")
        print("# project = desktop.open_project(r'C:\\path\\to\\your\\project.aedt')")
        print("#")
        print("# Maxwell 객체 생성:")
        print("# m2d = ansys.aedt.core.Maxwell2d(project=desktop, new_desktop=False)")
        print("# m3d = ansys.aedt.core.Maxwell3d(project=desktop, new_desktop=False)")
        print("=" * 60)
        
        return desktop
    else:
        print("❌ AEDT Desktop 연결에 실패했습니다.")
        print("\n🔍 문제 해결 방법:")
        print("1. Ansys AEDT가 설치되어 있는지 확인")
        print("2. AEDT 라이선스가 사용 가능한지 확인")
        print("3. 수동으로 AEDT를 실행한 후 다시 시도")
        return None

# 간단한 사용 함수들
def quick_connect():
    """빠른 연결 - 기존 세션 우선"""
    return try_connect_to_existing_desktop()

def force_new_session():
    """강제로 새 세션 생성"""
    try:
        return Desktop(
            specified_version=AEDT_VERSION,
            new_desktop_session=True,
            non_graphical=NG_MODE
        )
    except Exception as e:
        print(f"새 세션 생성 실패: {e}")
        return None


In [ ]:
get_aedt_processes_detailed()
m3d=ansys.aedt.core.Maxwell3d(new_desktop=False,version="2025.2")
designName=m3d.design_list
display(designName)
all_objects = m3d.modeler.object_names
for obj_name in all_objects:
    obj = m3d.modeler[obj_name]
materials=m3d.materials.material_keys
excitations=m3d.excitation_names
boundaryObjs=m3d.boundaries
bName = []
bproperties = []
for boundaryObj in boundaryObjs:
    bName.append(boundaryObj.name)
    bproperties.append(boundaryObj.properties)


In [ ]:
all_objects

### Plot

In [ ]:
py_vista_plot = m3d.post.plot_field(
    quantity="Mag_B", assignment='Rotor_Lamination_Primitive', plot_cad_objs=True, show=False
)
py_vista_plot.isometric_view = True
py_vista_plot.plot(
    export_image_path=os.path.join("D:", "Mag_B.jpg"), show=True)

In [ ]:
m3d.release_desktop()

## mesh

In [ ]:
from ansys.aedt.core.visualization.plot.pyvista import _parse_aedtplt 

pltPath=r"D:\KDH\gitPyAEDT\pyaedt\tests\system\visualization\example_models\T50\vector_field\SurfaceAcForceDensity.aedtplt"
vertices, faces, scalars, log=_parse_aedtplt(pltPath)
vertices=vertices[0]
faces=faces[0]
import matplotlib.pyplot as plt
import pyvista as pv
mesh = pv.PolyData(vertices, faces)
plotter = pv.Plotter()
plotter.add_mesh(mesh, show_edges=True, color="lightblue")
import tkinter as tk
import vtk

pv.set_jupyter_backend('trame')


plotter.add_mesh(mesh, show_edges=True)
# 노드 선택 활성화
def callback(point):
    print(f"선택한 노드 좌표: {point}")

plotter.enable_point_picking(callback=callback, use_mesh=True)

# 인터랙티브 플롯 실행
plotter.show()


# 2D Result Export

In [2]:
# e10 Model
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.

In [3]:
aedt=ansys.aedt.core.desktop

In [4]:
aedt_file=r"F:\KDH\Thesis\JEET\e10_tuto\e10_tutorial_ANSYSEM_2D.aedt"
# aedt_file=r"F:\KDH\Thesis\JEET\e10_tuto\MaxwellLabTutorial.aedt"
# aedt_file=r"F:\KDH\Thesis\JEET\e10\refModel\e10_UserRemesh_ANSYSEM_2D_2024.aedt"
m2d = ansys.aedt.core.Maxwell2d(
    project=aedt_file,
    version=AEDT_VERSION,
    new_desktop=False,
    non_graphical=NG_MODE
)

PyAEDT INFO: Parsing F:\KDH\Thesis\JEET\e10_tuto\e10_tutorial_ANSYSEM_2D.aedt.
PyAEDT INFO: Python version 3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]
PyAEDT INFO: PyAEDT version 0.13.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\user\AppData\Local\Temp\pyaedt_user_2675c68f-9464-487c-b835-8d1a8f96734d.log is enabled.
PyAEDT INFO: Log on AEDT is enabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: File F:\KDH\Thesis\JEET\e10_tuto\e10_tutorial_ANSYSEM_2D.aedt correctly loaded. Elapsed time: 0m 1sec
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: Found active AEDT gRPC session on port 50051
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v252\AnsysEM
PyAEDT INFO: Project e10_tutorial_ANSYSEM_2D set to active.
PyAEDT INFO: No consistent unique design is present. Inserting a new design.
PyAEDT INFO: Added de

In [ ]:
m2d.design_list

### get Design Variables


In [5]:
designName=m2d.design_list
display(designName)

m2d.set_active_design(designName[0])
allModelObj=m2d.modeler.model_objects

['Motor-CAD e10_tutorial',
 'Motor-CAD e10_tutorial_BPM_LabModel_1',
 'Motor-CAD e10_tutorial_BPM_LabModel_2']

PyAEDT INFO: Python version 3.8.10 (tags/v3.8.10:3d8993a, May  3 2021, 11:48:03) [MSC v.1928 64 bit (AMD64)]
PyAEDT INFO: PyAEDT version 0.13.0.
PyAEDT INFO: Returning found Desktop session with PID 42740!
PyAEDT INFO: Project e10_tutorial_ANSYSEM_2D set to active.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler2D class has been initialized!
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 1sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing objects from Data Model
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec


In [ ]:
materials=m2d.materials.material_keys

boundaryObjs=m2d.boundaries

bproperties = []
for boundaryObj in boundaryObjs:
    bName.append(boundaryObj.name)
    bproperties.append(boundaryObj.properties)


In [ ]:
exciteObj=m2d.excitation_objects
exciteObj[windingGroups[1]].properties

In [ ]:

for i, wg_name in enumerate(windingGroups):
    print(f"\n[{i+1}/{len(windingGroups)}] {wg_name} 업데이트 중...")
    
    try:
        # 현재 설정 가져오기
        current_props = exciteObj[wg_name].properties
        
        # PyAEDT 객체 업데이트
        exciteObj[wg_name].properties.update({'IsSolid': True})
        
        # AEDT API를 통한 업데이트
        oModule.EditWindingGroup(wg_name, [
            "NAME:" + wg_name,
            "Type:=", current_props.get('Type', 'Current'),
            "IsSolid:=", True,
            "Current:=", current_props.get('Current', '0A'),
            "Resistance:=", current_props.get('Resistance', '0ohm'),
            "Inductance:=", current_props.get('Inductance', '0H'),
            "Voltage:=", current_props.get('Voltage', '0V'),
            "ParallelBranchesNum:=", "ParallelPaths"
        ])
        
        print(f"  ✓ {wg_name} 업데이트 완료")
        
    except Exception as e:
        print(f"  ❌ {wg_name} 업데이트 실패: {e}")

print("\n✅ 모든 winding group 업데이트 완료!")

In [ ]:
m2d.close_desktop()

### Field Export in AEDT

## export_field_file

In [ ]:
fldFile=r"F:\KDH\Thesis\JEET\e10_tuto\allObject.aedtplt"
m2d.post.export_field_file(
    quantity="Mag_B",
    solution=
    output_file=fldFile,assignment="AllObjects"
)


PyAEDT INFO: Exporting Mag_B field. Be patient


INFO:Global:Exporting Mag_B field. Be patient


PyAEDT ERROR: **************************************************************


ERROR:Global:**************************************************************


PyAEDT ERROR:   File "C:\Program Files\Python38\lib\runpy.py", line 194, in _run_module_as_main


...

PyAEDT ERROR:     assignment = AllObjects 


ERROR:Global:    assignment = AllObjects 


PyAEDT ERROR: **************************************************************


ERROR:Global:**************************************************************


False

In [15]:
"""
Maxwell FLD 파일을 HDF5 형식으로 변환하는 스크립트
"""
import numpy as np
import pandas as pd
from pathlib import Path
import csv


def parse_fld_file(fld_file_path, header_lines=2):
    """
    FLD 파일을 파싱하여 데이터를 추출합니다.
    
    Parameters
    ----------
    fld_file_path : str or Path
        FLD 파일 경로
    header_lines : int
        건너뛸 헤더 라인 수 (기본값: 2)
        - 샘플 포인트 파일 사용 시: 1
        - 일반 export 시: 2
    
    Returns
    -------
    pandas.DataFrame
        파싱된 데이터|
    """
    fld_path = Path(fld_file_path)
    
    if not fld_path.exists():
        raise FileNotFoundError(f"FLD 파일을 찾을 수 없습니다: {fld_file_path}")
    
    # FLD 파일 읽기
    with open(fld_path, "r") as file:
        # 헤더 스킵
        for _ in range(header_lines):
            file.readline()
        
        # 데이터 파싱
        data_rows = []
        for line in file:
            # 공백 또는 탭으로 구분된 값 파싱
            tmp = line.strip().split()
            # 빈 탭 제거
            tmp = [element.replace("\t\t", "") for element in tmp if element]
            
            if len(tmp) > 1:  # 유효한 데이터 행만 추가
                data_rows.append(tmp)
    
    # NumPy 배열로 변환
    data_array = np.array(data_rows, dtype=float)
    
    # DataFrame 생성 (일반적으로 X, Y, Z, Field 값)
    if data_array.shape[1] == 4:
        # 스칼라 필드 (X, Y, Z, Field)
        columns = ['X', 'Y', 'Z', 'Field']
    elif data_array.shape[1] == 6:
        # 벡터 필드 (X, Y, Z, Fx, Fy, Fz)
        columns = ['X', 'Y', 'Z', 'Field_X', 'Field_Y', 'Field_Z']
    else:
        # 일반적인 경우
        columns = [f'Column_{i}' for i in range(data_array.shape[1])]
    
    df = pd.DataFrame(data_array, columns=columns)
    
    return df



In [ ]:
parse_fld_file(fldFile, header_lines=2)

UnicodeDecodeError: 'cp949' codec can't decode byte 0xf0 in position 0: illegal multibyte sequence

## to h5

In [ ]:
import h5py
import numpy as np
from pathlib import Path

def fld_to_h5_with_h5py(fld_file_path, output_h5_path=None, metadata=None, header_lines=2):
    """
    FLD 파일을 HDF5 형식으로 변환합니다 (h5py 사용).
    
    Parameters
    ----------
    fld_file_path : str or Path
        입력 FLD 파일 경로
    output_h5_path : str or Path, optional
        출력 HDF5 파일 경로
    metadata : dict, optional
        저장할 메타데이터
    header_lines : int
        건너뛸 헤더 라인 수
    
    Returns
    -------
    Path
        생성된 HDF5 파일 경로
    """
    fld_path = Path(fld_file_path)
    
    # 출력 파일 경로 설정
    if output_h5_path is None:
        output_h5_path = fld_path.with_suffix('.h5')
    else:
        output_h5_path = Path(output_h5_path)
    
    # FLD 파일 파싱
    print(f"FLD 파일 읽는 중: {fld_path}")
    df = parse_fld_file(fld_path, header_lines=header_lines)
    
    print(f"데이터 형태: {df.shape}")
    print(f"컬럼: {list(df.columns)}")
    
    # h5py로 HDF5 저장
    print(f"HDF5로 저장 중: {output_h5_path}")
    
    with h5py.File(output_h5_path, 'w') as h5f:
        # 데이터셋 생성 (압축 포함)
        for col in df.columns:
            h5f.create_dataset(
                col,
                data=df[col].values,
                compression='gzip',
                compression_opts=9  # 압축 레벨 (0-9)
            )
        
        # 메타데이터 저장
        if metadata:
            for key, value in metadata.items():
                h5f.attrs[key] = str(value)
        
        # DataFrame 정보 저장
        h5f.attrs['shape'] = df.shape
        h5f.attrs['columns'] = ','.join(df.columns)
    
    print(f"✓ 변환 완료!")
    return output_h5_path


def read_h5_with_h5py(h5_file_path):
    """
    h5py로 HDF5 파일에서 데이터를 읽어 DataFrame으로 변환
    
    Parameters
    ----------
    h5_file_path : str or Path
        HDF5 파일 경로
    
    Returns
    -------
    pandas.DataFrame
        필드 데이터
    """
    import pandas as pd
    
    with h5py.File(h5_file_path, 'r') as h5f:
        # 컬럼 정보 읽기
        columns = h5f.attrs['columns'].split(',')
        
        # 데이터 읽기
        data = {}
        for col in columns:
            data[col] = h5f[col][:]
        
        # DataFrame 생성
        df = pd.DataFrame(data)
        
        # 메타데이터 출력
        print("=== 메타데이터 ===")
        for key in h5f.attrs.keys():
            if key not in ['shape', 'columns']:
                print(f"{key}: {h5f.attrs[key]}")
    
    return df

In [ ]:
    
if __name__ == "__main__":
    # 예제 1: 기본 변환
    fldFile
    # 메타데이터 설정 (선택 사항)
    metadata = {
        "source": "Maxwell 2D",
        "quantity": "Mag_B",
        "solution": "Setup1 : LastAdaptive",
        "units": "Tesla",
        "coordinate_system": "Global",
        "date": "2025-01-07"
    }
    
    # FLD → H5 변환
    h5_file = fld_to_h5_with_h5py(
        fld_file_path=fldFile,
        metadata=metadata,
        header_lines=2  # 일반 export는 2, sample points 사용 시 1
    )
    

In [ ]:
from ansys.aedt.core.visualization.plot.pyvista import ModelPlotter


import pyvista as pv
model_plotter = ModelPlotter()    
model_plotter.add_field_from_file(fldFile)
# parse_fld_file(fldFile, header_lines=2)

# Backup

In [ ]:
# AEDT Maxwell GUI 실행
import pyaedt
print("=== ANSYS Electronics Desktop (AEDT) Maxwell GUI 실행 ===")

# GUI 모드로 AEDT 실행
print("Maxwell 2D GUI를 실행합니다...")

try:
    # GUI 모드로 Maxwell 2D 새 프로젝트 생성
    m2d_gui = pyaedt.Maxwell2d(
        non_graphical=False,  # GUI 모드로 실행
        new_desktop_session=True,  # 새로운 데스크톱 세션 시작
        close_on_exit=False,  # 종료 시 자동으로 닫지 않음
        student_version=False
    )
    
    print(f"✓ Maxwell 2D GUI가 성공적으로 실행되었습니다!")
    print(f"  - 프로젝트명: {m2d_gui.project_name}")
    print(f"  - 디자인명: {m2d_gui.design_name}")
    print(f"  - AEDT 버전: {m2d_gui.aedt_version_id}")
    print(f"  - 솔루션 타입: {m2d_gui.solution_type}")
    
    # GUI 창 정보
    print(f"\n📺 AEDT Maxwell GUI 창이 열렸습니다!")
    print("  - Maxwell 2D TransientXY 환경")
    print("  - 새로운 프로젝트로 시작")
    print("  - 모델링, 해석, 포스트프로세싱 가능")
    
    # 객체를 전역 변수로 저장
    globals()['maxwell_gui'] = m2d_gui
    
except Exception as e:
    print(f"❌ GUI 실행 중 오류 발생: {e}")
    print("다음 사항을 확인해주세요:")
    print("  1. ANSYS Electronics Desktop이 설치되어 있는지")
    print("  2. 라이센스가 유효한지")
    print("  3. 다른 AEDT 세션이 실행 중인지")

print("\n🚀 Maxwell GUI가 준비되었습니다!")